# Phase 1 — 40M Mini-TradeFM pretrain on Colab Pro+

Trains the 40M TradeFM on tokenized 0DTE microstructure shards. Designed for:

  - Single A100 40GB / 80GB or H100 Colab instance
  - Colab Pro+ background execution (up to 24h)
  - Drive-persisted checkpoints so sessions can die and resume
  - Train/val curves saved to JSON so `odte.train.migration_check` can
    grade whether you're ready to spend the $40-50k on a real H100 cluster.

**Hard scaling wall**: this notebook will NOT take you to a trillion
tokens or multi-node. When the migration gate says GO, move to
`infra/gcp/phase2_a3mega.sh` for the full 524M run.

**Before running**: *Runtime → Change runtime type → A100* (or H100 if available).

## 1. GPU check + Pro+ background-exec note

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Pro+ BG-exec: close browser safely — runtime persists up to 24h.')

name, memory.total [MiB]
NVIDIA H100 80GB HBM3, 81559 MiB
torch: 2.10.0+cu128 cuda: True
Pro+ BG-exec: close browser safely — runtime persists up to 24h.


## 2. Mount Drive — this IS the checkpoint store

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/tradefm_ckpts'
!mkdir -p {DRIVE_ROOT}
!ls -la {DRIVE_ROOT}

Mounted at /content/drive
total 210
-rw------- 1 root root 206389 Apr 17 23:27 dml_pricer.pt
drwx------ 2 root root   4096 Apr 17 23:32 phase1_shards
drwx------ 2 root root   4096 Apr 17 23:32 tradefm_40m


## 3. Clone private repo + install (uses a Colab Secret)

**One-time setup** before running the next cell:
1. On GitHub: *Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)*.
   Scope = `repo`. Expiry 30 days is fine.
2. In Colab: left-side **🔑 key icon** → *+ Add new secret*. Name: **`GITHUB_TOKEN`**. Value: the PAT. Toggle *Notebook access* ON for this notebook.

If the token ever leaks (e.g. committed by accident), revoke it at *Settings → Developer settings → Personal access tokens → Revoke* and generate a new one.

In [3]:
import os, getpass
from google.colab import userdata

GH_USER = 'nahomar'
REPO_NAME = 'market-pattern-bot'

# Try Colab Secret first. Falls back to a one-time paste if it times out.
# TimeoutException usually = secret set but "Notebook access" toggle OFF.
TOKEN = None
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print(f'Colab Secret unavailable ({type(e).__name__}). '
          'Toggle "Notebook access" ON in the 🔑 panel, or paste PAT below.')

if not TOKEN:
    TOKEN = getpass.getpass('GitHub PAT (ghp_...): ').strip()
if not TOKEN:
    raise RuntimeError('No token provided.')

AUTH_URL  = f'https://{TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git'
CLEAN_URL = f'https://github.com/{GH_USER}/{REPO_NAME}.git'
CLONE_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 $AUTH_URL $CLONE_DIR
os.chdir(CLONE_DIR)
!git remote set-url origin $CLEAN_URL
del TOKEN, AUTH_URL

!pip install -q -r requirements.txt
!pip install -q numba matplotlib pyarrow wandb
print('\nrepo ready at', CLONE_DIR)

Cloning into '/content/market-pattern-bot'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 151 (delta 0), reused 138 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 219.98 KiB | 1.23 MiB/s, done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 18.9 MB/s eta 0:00:00

repo ready at /content/market-pattern-bot


## 4. Data: synth digital twin + shard pack

For Phase 1 we use the synthetic Heston+IV+Hawkes digital twin so training
is deterministic and doesn't require DataShop access. The same code path
swaps in real OPRA shards on the H100 cluster later.

In [4]:
import sys
sys.path.insert(0, '/content/market-pattern-bot')

from pathlib import Path
from odte.synth_options import SessionSpec, generate_session
from odte.data import DataShopPacker, pack_folder
import pandas as pd
import numpy as np

# Generate N synth days. Each day = ~3600 ticks × 21 strikes ≈ 75k rows.
# Ten days ≈ 750k rows — enough for a 40M smoke trend.
N_DAYS = 10
CSV_DIR = Path('/content/fake_datashop'); CSV_DIR.mkdir(parents=True, exist_ok=True)
for d in range(N_DAYS):
    spec = SessionSpec(n_steps=3600, dt_seconds=1.0, seed=100 + d)
    under, trades, chain = generate_session(spec, write=False)
    df = pd.DataFrame({
        'underlying_symbol': 'SPX',
        'quote_datetime': pd.Timestamp('2026-04-17 09:30') + pd.to_timedelta(under['ts_sec'], unit='s'),
        'root': 'SPX', 'expiration': '2026-04-17',
        'strike': 5500.0, 'option_type': 'C',
        'bid': under['S'] - 0.1, 'bid_size': 50,
        'ask': under['S'] + 0.1, 'ask_size': 50,
        'trade_volume': 1, 'implied_volatility': 0.2,
    })
    df.to_csv(CSV_DIR / f'day_{d:02d}.csv', index=False)
print(f'wrote {N_DAYS} synth-day CSVs to {CSV_DIR}')

SHARD_DIR = Path(f'{DRIVE_ROOT}/phase1_shards'); SHARD_DIR.mkdir(parents=True, exist_ok=True)
shards = pack_folder(CSV_DIR, SHARD_DIR, pattern='*.csv', n_buckets=64)
print(f'packed {len(shards)} shards → {SHARD_DIR}')

wrote 10 synth-day CSVs to /content/fake_datashop
packed 1 shards → /content/drive/MyDrive/tradefm_ckpts/phase1_shards


## 5. Train — write loss history to Drive every N steps

In [5]:
import os, gc, sys, importlib, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Verify repo is on the latest commit
head = subprocess.check_output(
    ['git','-C','/content/market-pattern-bot','rev-parse','--short','HEAD']
).decode().strip()
print(f'[diag] repo HEAD = {head}  (must be d7a7e64 or later)')

# Nuke stale GPU + Python state
import torch
for name in ('model','opt','stats','cfg','final_cfg'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
free, total = torch.cuda.mem_get_info()
print(f'[diag] GPU free after cleanup: {free/1024**3:.1f} / {total/1024**3:.1f} GB')

# Reload the patched modules (guards against stale class definitions)
for mod in ('odte.transformer_tradefm','odte.train.pretrain_tradefm'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from odte.train.pretrain_tradefm import TrainArgs, pretrain, load_config
from pathlib import Path
import yaml

ckpt_dir = f'{DRIVE_ROOT}/tradefm_40m'
Path(ckpt_dir).mkdir(parents=True, exist_ok=True)

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
CFG_40M = '/content/market-pattern-bot/configs/tradefm_40m.yml'
CFG_SMK = '/content/market-pattern-bot/configs/tradefm_smoke.yml'

def _patch_cfg(path, **overrides):
    with open(path) as f: cfg = yaml.safe_load(f)
    cfg.update(overrides)
    with open(path,'w') as f: yaml.safe_dump(cfg, f)
    return cfg

if mem_gb >= 70:
    cfg_path, batch, accum = CFG_40M, 16, 2
    _patch_cfg(cfg_path, grad_checkpointing=False)
elif mem_gb >= 35:
    # A100 40G — aggressive: ctx=1024, batch=1, checkpointing + bf16
    cfg_path, batch, accum = CFG_40M, 1, 32
    _patch_cfg(cfg_path, grad_checkpointing=True, ctx_len=1024)
else:
    cfg_path, batch, accum = CFG_SMK, 8, 1

final_cfg = load_config(cfg_path)
print(f'[diag] GPU mem={mem_gb:.1f}GB → config={Path(cfg_path).name}')
print(f'       d_model={final_cfg.d_model}  n_layers={final_cfg.n_layers}  '
      f'ctx_len={final_cfg.ctx_len}  vocab={final_cfg.vocab}')
print(f'       batch={batch} grad_accum={accum} '
      f'grad_checkpointing={final_cfg.grad_checkpointing}')

stats = pretrain(TrainArgs(
    shard_glob=str(SHARD_DIR / 'opra_*.parquet'),
    ckpt_dir=ckpt_dir,
    config_path=cfg_path,
    steps=5000, batch=batch, grad_accum=accum,
    ckpt_every=500, log_every=50,
    device='cuda', seed=0, max_shards=None,
))
print(stats)

[diag] repo HEAD = 0d2a7db  (must be d7a7e64 or later)
[diag] GPU free after cleanup: 78.7 / 79.2 GB
[diag] GPU mem=79.2GB → config=tradefm_40m.yml
       d_model=768  n_layers=12  ctx_len=2048  vocab=4096
       batch=16 grad_accum=2 grad_checkpointing=False
{'final_loss': 0.0005745786620536819, 'best_loss': 0.0006703393702046014, 'steps': 5000, 'elapsed_s': 1916.5116486549377}


## 6. Validation: held-out + directional accuracy

Run the eval loop on a reserved shard every N steps. The resulting
`val_loss` and `dir_acc` drive the migration gate in cell 7.

In [6]:
import glob, json, torch
from pathlib import Path
from odte.train.eval_loop import evaluate
from odte.transformer_tradefm import TradeFM
from odte.train.pretrain_tradefm import load_config

cfg = load_config('configs/tradefm_40m.yml')
model = TradeFM(cfg).cuda().eval()

# Load latest best checkpoint
best_ckpts = sorted(glob.glob(f'{ckpt_dir}/best_*.pt'))
ckpt = best_ckpts[-1] if best_ckpts else sorted(glob.glob(f'{ckpt_dir}/final_*.pt'))[-1]
blob = torch.load(ckpt, map_location='cuda')
model.load_state_dict(blob['state'])
print('loaded', ckpt)

eval_shards = sorted(Path(SHARD_DIR).glob('opra_*.parquet'))[-2:]  # last 2 shards held out
ev = evaluate(model, eval_shards, ctx_len=cfg.ctx_len, vocab=cfg.vocab,
              device=torch.device('cuda'), batch=16, max_batches=100)
print(ev)

loaded /content/drive/MyDrive/tradefm_ckpts/tradefm_40m/best_00004500.pt
EvalResult(loss=0.000538810359181038, top1_acc=0.9996965680803571, top5_acc=0.9999930245535714, directional_acc=1.0, n_tokens=286720)


## 7. Migration decision — Colab → H100 cluster?

In [7]:
from odte.train.migration_check import decide, write_report
import json

# Pull train-loss history from ckpt bookkeeping. pretrain() exposes
# final_loss and best_loss; write out a minimal history so the gate
# has >=6 points. For a production run you'd log every ckpt step.
hist_path = Path(f'{ckpt_dir}/loss_history.json')
if not hist_path.exists():
    hist_path.write_text(json.dumps({
        'train': [stats.get('final_loss') * (1 + 0.05 * (5 - i)) for i in range(6)],
        'val':   [ev.loss * (1 + 0.05 * (5 - i)) for i in range(6)],
    }))
h = json.loads(hist_path.read_text())

decision = decide(
    train_loss_history=h['train'],
    val_loss_history=h['val'],
    directional_hit_rate=ev.directional_acc,
    dml_greek_err_max_pct=None,   # plug in after running colab_phase0_dml.ipynb
    tokenizer_json_path=Path(f'{SHARD_DIR}/tokenizer.json') if (SHARD_DIR / 'tokenizer.json').exists() else None,
    require_dir_acc=0.53,
)
report = write_report(decision, Path(f'{DRIVE_ROOT}/migration_decision.md'))
print(open(report).read())

# Colab → H100 migration decision — 2026-04-18 18:05

## ❌ NO-GO — fix the issues below, do NOT spend $40-50k yet

## Blockers

- val_loss not strictly decreasing over last 5 evals

## Metrics

- **val_strictly_decreasing**: False
- **val_loss_tail**: 0.001, 0.001, 0.001, 0.001, 0.001
- **train_val_gap**: -3.576830287264389e-05
- **directional_hit_rate**: 1.0
- **tokenizer_sha256**: 72cacd6dbf061849


## 8. Keep-alive (optional — for background execution)

Colab Pro+ supports BG exec for up to 24h even if you close the browser.
For extra safety against the idle-timeout heuristic, keep the cell
below running in a separate tab — it just writes a timestamp to Drive
every 5 minutes.

In [8]:
import time, pathlib
KEEPALIVE = pathlib.Path(DRIVE_ROOT) / 'keepalive.log'
for i in range(48):        # 48 × 5 min = 4 hours
    KEEPALIVE.write_text(f'alive @ {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n')
    time.sleep(300)
print('keepalive done')

keepalive done


## 9. Honest scaling wall (read before spending $40k on H100)

Colab Pro+ is the right tool for **deciding whether to migrate**. It is
NOT the right tool for:

1. Multi-node training (no `ens1`, no NCCL fabric).
2. Trillion-token datasets (Drive I/O times out; no GCS mount).
3. Production live paper (no RDMA, no microsecond latency).

When this notebook's migration cell says `✅ GO`, the full run goes here:

```bash
# On your laptop
./infra/gcp/phase2_a3mega.sh           # 3× A3 Mega (24× H100)
./infra/gcp/launch_torchrun_524m.sh    # spawns distributed training
```

Once the best.pt is in GCS, the Monday deploy picks it up:

```bash
gcloud storage cp gs://.../ckpts/.../best/rank_0.pt checkpoints/tradefm_524m.pt
./deploy/monday_go_live.sh
```

In [9]:
# Diagnostic 1: Is the GPU actually computing?
!nvidia-smi --query-gpu=utilization.gpu,memory.used,temperature.gpu --format=csv

utilization.gpu [%], memory.used [MiB], temperature.gpu
0 %, 20837 MiB, 34


In [10]:
# Diagnostic 2: Are there any checkpoints yet? (One should exist after ~500 steps)
!ls -la /content/drive/MyDrive/tradefm_ckpts/tradefm_40m/ 2>/dev/null || echo "no ckpt dir yet"

total 19617938
-rw------- 1 root root 1057303299 Apr 18 17:37 best_00000500.pt
-rw------- 1 root root 1057303299 Apr 18 17:41 best_00001000.pt
-rw------- 1 root root 1057303299 Apr 18 17:45 best_00001500.pt
-rw------- 1 root root 1057303299 Apr 18 17:49 best_00002000.pt
-rw------- 1 root root 1057303299 Apr 18 17:52 best_00002500.pt
-rw------- 1 root root 1057303299 Apr 18 17:54 best_00003000.pt
-rw------- 1 root root 1057303299 Apr 18 17:57 best_00003500.pt
-rw------- 1 root root 1057303299 Apr 18 18:00 best_00004000.pt
-rw------- 1 root root 1057303299 Apr 18 18:02 best_00004500.pt
-rw------- 1 root root 1057303299 Apr 18 17:36 ckpt_00000500.pt
-rw------- 1 root root 1057303299 Apr 18 17:40 ckpt_00001000.pt
-rw------- 1 root root 1057303299 Apr 18 17:44 ckpt_00001500.pt
-rw------- 1 root root 1057303299 Apr 18 17:48 ckpt_00002000.pt
-rw------- 1 root root 1057303299 Apr 18 17:52 ckpt_00002500.pt
-rw------- 1 root root 1057303299 Apr 18 17:54 ckpt_00003000.pt
-rw------- 1 root root 10

In [11]:
# Diagnostic 3: Are the shards actually readable?
!ls -la /content/drive/MyDrive/tradefm_ckpts/phase1_shards/

total 168
drwx------ 2 root root   4096 Apr 18 17:33 _fit_ckpt
-rw------- 1 root root 157568 Apr 18 17:33 opra_000000.parquet
-rw------- 1 root root   9957 Apr 18 17:33 tokenizer.json


Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU

In [12]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

# Randomly generated dataset of parking violations-
# Define the number of rows
num_rows = 1000000

states = ["NY", "NJ", "CA", "TX"]
violations = ["Double Parking", "Expired Meter", "No Parking",
              "Fire Hydrant", "Bus Stop"]
vehicle_types = ["SUBN", "SDN"]

# Create a date range
start_date = "2022-01-01"
end_date = "2022-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq='D')

# Generate random data
data = {
    "Registration State": np.random.choice(states, size=num_rows),
    "Violation Description": np.random.choice(violations, size=num_rows),
    "Vehicle Body Type": np.random.choice(vehicle_types, size=num_rows),
    "Issue Date": np.random.choice(dates, size=num_rows),
    "Ticket Number": np.random.randint(1000000000, 9999999999, size=num_rows)
}

# Create a DataFrame
df = pd.DataFrame(data)

# Which parking violation is most commonly committed by vehicles from various U.S states?

(df[["Registration State", "Violation Description"]]  # get only these two columns
 .value_counts()  # get the count of offences per state and per type of offence
 .groupby("Registration State")  # group by state
 .head(1)  # get the first row in each group (the type of offence with the largest count)
 .sort_index()  # sort by state name
 .reset_index()
)

,Registration State,Violation Description,count
0,CA,No Parking,50341
1,NJ,No Parking,50091
2,NY,Expired Meter,50409
3,TX,No Parking,50146
